In [1]:
import numpy as np
import pandas as pd
import torch

import sys
sys.path.append('../')
from utilities import binning_equal_q

In [2]:
def infer_fields(data, counts):
    
    fields = np.zeros([7,20])
    for pos in range(7):
        for color in range(20):
            fields[pos,color] = (counts*((data[:,pos] == color).astype('int'))).sum() 

    print(counts.sum(), fields.sum(1))

    fields = fields[:,:] / fields[:,0][:,np.newaxis]
    return torch.tensor(np.log(fields))

def sorted_log10p_vector_from_fields(fields):
    
    lengths = [torch.arange(20, dtype=torch.int8) for i in range(7)]
    all_seq = torch.cartesian_prod(*lengths)
    
    all_seq = all_seq.long()
    
    # generate p_vector
    p_vector = torch.zeros(20**7,dtype=torch.float32)

    for i in range(7):
        p_vector += fields[i,all_seq[:,i]]
        print(i)
    
    p_vector = torch.exp(p_vector)
    
    Z = torch.exp(fields).sum(1).prod()
    
    p_vector /= Z
    print(p_vector.sum())
    
    return torch.log(p_vector).sort()[0] / np.log(10)

def log10q_vector_func(data, fields):
    
    # generate q_vector
    q_vector = torch.zeros(len(data))

    for i in range(7):
        q_vector += fields[i,data[:,i]]
        print(i)

    q_vector = torch.exp(q_vector)

    Z = torch.exp(fields).sum(1).prod()

    q_vector /= Z

    print(q_vector.sum())
    return np.log10(q_vector.numpy())

In [22]:
t = 256

In [23]:
data = pd.read_csv('../data/Byrne.csv',index_col=0).query('T0 > 0')
data = data.sort_values('T0', ascending=False)
data = data.iloc[1:,:]

np.random.seed(0)
data['sample'] = np.random.binomial(data['T0'], p=np.array([1/t]))
data = data.query('sample > 0')

counts = data['sample'].to_numpy()
data = data.iloc[:,:7].to_numpy()

In [24]:
fields = infer_fields(data, counts)

47622 [47622. 47622. 47622. 47622. 47622. 47622. 47622.]


In [25]:
sorted_log10p_vector = sorted_log10p_vector_from_fields(fields)

0
1
2
3
4
5
6
tensor(1.)


In [26]:
log10q_vector = log10q_vector_func(data, fields)
argsort = np.argsort(log10q_vector)
sorted_log10q_vector = log10q_vector[argsort][::-1]
counts = counts[argsort][::-1]

0
1
2
3
4
5
6
tensor(0.0002)


In [27]:
bins=80
df_bins = binning_equal_q(sorted_log10p_vector, sorted_log10q_vector, counts, bins=bins, writefolder=False)#'results_ByrneT0_IM', step=10)
df_bins.to_csv('df_bins_Byrne_IM_%dt.csv'%t)

0
tensor(1279999872) tensor(1279718890)
elements in the bin: 280982
nonzeros: 586
1
tensor(1279718890) tensor(1279278406)
elements in the bin: 440484
nonzeros: 586
2
tensor(1279278406) tensor(1278707078)
elements in the bin: 571328
nonzeros: 586
3
tensor(1278707078) tensor(1278089984)
elements in the bin: 617094
nonzeros: 586
4
tensor(1278089984) tensor(1277410646)
elements in the bin: 679338
nonzeros: 586
5
tensor(1277410646) tensor(1276629349)
elements in the bin: 781297
nonzeros: 586
6
tensor(1276629349) tensor(1275718642)
elements in the bin: 910707
nonzeros: 586
7
tensor(1275718642) tensor(1274777269)
elements in the bin: 941373
nonzeros: 586
8
tensor(1274777269) tensor(1273703921)
elements in the bin: 1073348
nonzeros: 586
9
tensor(1273703921) tensor(1272502682)
elements in the bin: 1201239
nonzeros: 586
10
tensor(1272502682) tensor(1271294001)
elements in the bin: 1208681
nonzeros: 586
11
tensor(1271294001) tensor(1269927821)
elements in the bin: 1366180
nonzeros: 586
12
tensor(

In [28]:
t

256

## to find P(seq) of TSKATIA

In [8]:
pseq_vec = []
for t in 2**np.arange(9):
    
    data = pd.read_csv('../data/Byrne.csv',index_col=0).query('T0 > 0')
    data = data.sort_values('T0', ascending=False)
    data = data.iloc[1:,:]

    np.random.seed(0)
    data['sample'] = np.random.binomial(data['T0'], p=np.array([1/t]))
    data = data.query('sample > 0')

    counts = data['sample'].to_numpy()
    data = data.iloc[:,:7].to_numpy()
    
    fields_numpy = infer_fields(data, counts).numpy()
    
    pseq = (np.exp(fields_numpy[0,16]+fields_numpy[1,15]+fields_numpy[2,8]+fields_numpy[3,0]+fields_numpy[4,16]+fields_numpy[5,7]+fields_numpy[6,0])/np.exp(fields_numpy).sum(1).prod())
    pseq_vec.append(pseq)
pseq_vec = np.array(pseq_vec)

12195473 [12195473. 12195473. 12195473. 12195473. 12195473. 12195473. 12195473.]
6098191 [6098191. 6098191. 6098191. 6098191. 6098191. 6098191. 6098191.]
3048615 [3048615. 3048615. 3048615. 3048615. 3048615. 3048615. 3048615.]
1524221 [1524221. 1524221. 1524221. 1524221. 1524221. 1524221. 1524221.]
762712 [762712. 762712. 762712. 762712. 762712. 762712. 762712.]
381370 [381370. 381370. 381370. 381370. 381370. 381370. 381370.]
190542 [190542. 190542. 190542. 190542. 190542. 190542. 190542.]
95467 [95467. 95467. 95467. 95467. 95467. 95467. 95467.]
47622 [47622. 47622. 47622. 47622. 47622. 47622. 47622.]


In [10]:
np.savetxt('Byrne_TSKATIA_p_seq_different_t.csv',pseq_vec)